# Formulación del problema y descripción de los datos

## Análisis estratégico con enfoque Google

**Resultado ideal (objetivo del cliente).**
Un banco que vende depósitos a plazo por telemarketing quiere gastar sus llamadas donde más rinden.
Hoy llama a una lista amplia de clientes y solo uno de cada ocho contrata. El resultado ideal es
decidir, antes de marcar el número, a quién conviene llamar, de modo que cada llamada tenga más
probabilidad de terminar en una contratación.

**Objetivo del modelo (acción concreta a predecir).**
Predecir si un cliente contratará el depósito a plazo (`yes`) o no (`no`) usando únicamente
información conocida **antes** de la llamada: perfil del cliente, situación financiera, historial de
campañas previas y datos de la campaña en curso.

**Filtro de formulación: por qué clasificación binaria.**
La decisión del banco es binaria (llamar o no llamar) y el desenlace también (contrata o no contrata).
No hay una cantidad continua que estimar ni más de dos desenlaces con acciones distintas, de modo que
la clasificación binaria es la formulación que corresponde. Como la clase positiva es minoritaria
(alrededor del 12 %), la accuracy por sí sola no sirve para juzgar un modelo.

**Métricas de éxito.**

| Métrica | Para qué sirve en este problema |
|---|---|
| F1 de la clase positiva | Equilibra a cuántos contratadores se encuentra (recall) y cuántas llamadas se desperdician (precisión) |
| ROC-AUC | Mide qué tan bien el modelo ordena a los clientes de más a menos probables, independiente del umbral |
| Recall de la clase positiva | Qué fracción de los clientes que sí contratarían queda dentro de la lista de llamadas |
| Tiempo de entrenamiento y de predicción | Costo computacional de cada familia de modelos, que condiciona cuál conviene usar en producción |

**Metas medibles del experimento.** Para poder decir al final si el experimento tuvo éxito, fijé metas
concretas antes de ajustar ningún modelo. Las mido solo sobre el conjunto de prueba, que no interviene
en ningún ajuste.

| Meta | Umbral | Por qué ese umbral |
|---|---|---|
| ROC-AUC | 0,75 o más | Un modelo sin información obtiene 0,50; 0,75 es una capacidad de ordenamiento claramente útil |
| F1 de la clase positiva | 0,40 o más | El modelo trivial obtiene 0; esta cifra obliga a encontrar contratadores sin llenar la lista de falsos positivos |
| Captura en el 20 % superior | 40 % o más de los contratadores | El doble de lo que lograría una lista al azar (20 %) |
| Sobreajuste | Brecha de F1 entre entrenamiento y prueba de 0,10 o menos | Un modelo que memoriza no se sostendrá con clientes nuevos |
| Costo computacional | Entrenar en menos de 60 s y predecir 9.043 clientes en menos de 1 s | Permite reentrenar con cada campaña y puntuar listas completas sin infraestructura especial |

En el notebook 05 verifiqué cada meta para cada uno de los cuatro modelos.

**Herramientas.** Usé Python 3, scikit-learn (SVM, árbol de decisión, Random Forest), XGBoost (boosting),
pandas, matplotlib y seaborn, ejecutados en Google Colab.

**Una variable que no puede usarse.** El conjunto incluye `duration`, la duración de la llamada en
segundos. Esa cifra solo se conoce cuando la llamada ya terminó, y una llamada larga suele ser
consecuencia de que el cliente estaba interesado. Usarla como predictora equivale a usar el
resultado para predecir el resultado, así que la excluí del experimento. En el notebook 02 cuantifiqué
cuánta información contiene y por qué su inclusión inflaría las métricas.

## Objetivos del experimento

**Objetivo general.**
Comparar cuatro modelos de clasificación (SVM, árbol de decisión, Random Forest y XGBoost) para
predecir si un cliente contratará un depósito a plazo, evaluando tanto su rendimiento como su costo
computacional, y justificar cuál conviene usar.

**Objetivos específicos.**

1. Caracterizar el conjunto de datos, describiendo las variables y la distribución de la variable
   objetivo, e identificar outliers, correlaciones y relaciones con la contratación.
2. Preparar los datos con un tratamiento diferenciado: escalar solo para el SVM, que depende de
   distancias, y dejar sin escalar los modelos basados en árboles, y dividir con estratificación y
   semilla fija.
3. Entrenar y ajustar un SVM, justificando el kernel y ajustando `C` y `gamma` por validación cruzada.
4. Entrenar y ajustar un árbol de decisión, controlando el sobreajuste con `max_depth` y
   `min_samples_leaf`, e interpretar la importancia de las variables.
5. Entrenar y ajustar un Random Forest, usando la evaluación out-of-bag para ajustar hiperparámetros.
6. Entrenar y ajustar un XGBoost, controlando `learning_rate`, `n_estimators` y el sobreajuste con
   early stopping.
7. Medir el tiempo de entrenamiento y de predicción de cada modelo, compararlos con una misma tabla
   de métricas y justificar la selección del mejor modelo.

## Carga del conjunto de datos

Usé el conjunto de Moro, Rita y Cortez (2014), *A data-driven approach to predict the success of bank
telemarketing*, publicado en **Decision Support Systems** y disponible en el repositorio UCI Machine
Learning Repository (conjunto *Bank Marketing*). Reúne los contactos telefónicos de campañas de
marketing directo de un banco portugués. Lo leí desde una URL pública para que no requiera descarga
manual ni credenciales. Como el archivo viene comprimido dos veces, la función de carga abre un `.zip`
dentro de otro `.zip` y guarda una copia local para no descargarlo en cada notebook.

### Entorno y persistencia de resultados

En la celda siguiente fijé tres cosas que condicionan la reproducibilidad del experimento.

**Semilla fija.** Usé `RANDOM_STATE = 42` en la partición train/test y en el ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Escribí los resultados en `MyDrive/bank_marketing` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. El notebook 03 deja las particiones
preparadas y los notebooks 04 y 05 las consumen tal cual, de modo que todos operan sobre exactamente
los mismos datos. Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir
en una ubicación volátil. Fuera de Colab no hace falta Drive, y los resultados se escriben en la
carpeta del repositorio (los gráficos en `splits/`).

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. La invoco antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib y el
archivo saldría en blanco.

In [1]:
import os, io, time, json, zipfile, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedó montado. Vuelve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/bank_marketing"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Gráfico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando localmente")
print("Ruta de trabajo:", RUTA)

Google Drive no disponible: guardando localmente
Ruta de trabajo: C:\Users\USUARIO\Documents\6_SEMESTRE\MACHINE LEARNING\Bank_Marketing\bank_marketing


In [2]:
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
ARCHIVO_LOCAL = os.path.join(RUTA, "bank-full.csv")

def cargar_datos():
    if os.path.exists(ARCHIVO_LOCAL):
        return pd.read_csv(ARCHIVO_LOCAL, sep=";")
    with urllib.request.urlopen(URL, timeout=120) as respuesta:
        externo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
    interno = zipfile.ZipFile(io.BytesIO(externo.read("bank.zip")))
    datos = pd.read_csv(interno.open("bank-full.csv"), sep=";")
    datos.to_csv(ARCHIVO_LOCAL, sep=";", index=False)
    return datos

df = cargar_datos()
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

Filas: 45211 | Columnas: 17


In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## Descripción de variables

**Variable objetivo:** `y`, con dos clases: `yes` (el cliente contrató el depósito) y `no`.

Las variables predictoras se agrupan en cuatro familias:

| Familia | Variables | Tipo | Significado en el dominio |
|---|---|---|---|
| Perfil del cliente | `age`, `job`, `marital`, `education` | Numérica / categórica nominal | Edad, ocupación, estado civil y nivel de estudios |
| Situación financiera | `default`, `balance`, `housing`, `loan` | Categórica binaria / numérica | Si tiene mora, saldo anual promedio, crédito de vivienda y crédito personal |
| Contacto de la campaña | `contact`, `day`, `month`, `duration` | Categórica / numérica | Canal, día y mes del último contacto y duración de la llamada |
| Historial de campañas | `campaign`, `pdays`, `previous`, `poutcome` | Numérica discreta / categórica | Contactos en esta campaña, días desde la campaña anterior, contactos previos y resultado previo |

Dejé `duration` fuera del experimento por la razón expuesta arriba. Las otras 15 variables son las
predictoras.

In [4]:
print("=== Tipos de dato ===")
print(df.dtypes.value_counts(), "\n")

categoricas = df.select_dtypes(exclude="number").columns.drop("y").tolist()
print("=== Cardinalidad de las variables categóricas ===")
print(df[categoricas].nunique().sort_values(ascending=False), "\n")

print("=== Variable objetivo ===")
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True).round(4))

=== Tipos de dato ===
str      10
int64     7
Name: count, dtype: int64 

=== Cardinalidad de las variables categóricas ===
job          12
month        12
education     4
poutcome      4
marital       3
contact       3
default       2
housing       2
loan          2
dtype: int64 

=== Variable objetivo ===
y
no     39922
yes     5289
Name: count, dtype: int64
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


**Lectura.** El conjunto tiene 45.211 clientes y 16 variables predictoras, 7 numéricas y 9 categóricas.
Solo 5.289 contrataron el depósito (11,7 %), una razón de 7,55 clientes que no contratan por cada uno
que sí. Esa proporción manda sobre casi todas las decisiones siguientes. Un modelo que respondiera
siempre `no` acertaría el 88,3 % de las veces sin aprender nada, por lo que la accuracy no puede ser la
métrica de éxito. `job` y `month` son las variables con más categorías, doce cada una.

In [5]:
desconocidos = (df[categoricas] == "unknown").sum()
tabla_desconocidos = pd.DataFrame({
    "unknown": desconocidos,
    "porcentaje": (100 * desconocidos / len(df)).round(2),
}).query("unknown > 0").sort_values("unknown", ascending=False)
print(tabla_desconocidos, "\n")

sin_contacto_previo = int((df["pdays"] == -1).sum())
print("Filas con pdays = -1 (sin contacto previo):", sin_contacto_previo,
      f"({100 * sin_contacto_previo / len(df):.2f} %)")
print("Valores nulos reales:", int(df.isna().sum().sum()))

           unknown  porcentaje
poutcome     36959       81.75
contact      13020       28.80
education     1857        4.11
job            288        0.64 

Filas con pdays = -1 (sin contacto previo): 36954 (81.74 %)


Valores nulos reales: 0


**Lectura.** No hay nulos reales, pero sí categorías `unknown`. En `poutcome` es el 81,75 % de las
filas y coincide casi exacto con los clientes que nunca fueron contactados antes (`pdays = -1`,
81,74 %). En `contact` es el 28,8 %. En `education` y `job` es marginal (4,11 % y 0,64 %). Para
`poutcome` y `contact` el desconocido no es un dato faltante por error sino una condición del cliente
(no hubo campaña previa, o el canal no quedó registrado), así que se tratará como una categoría más.